# Testing in Data Science

For this notebook, we recommend splitting the VS Code editor. You can drag this notebook or the exercise files to a second panel while you work.

This notebook lives in `03-testing-in-data-science/`:

<pre>
03-testing-in-data-science/
├── 03-testing-in-data-science.ipynb
├── data/
│   └── measurementRectanglesMissing.csv
├── src/
│   └── data_pipeline/
│      ├── __init__.py
│      ├── imputation_solution.py
│      ├── imputation.py
│      ├── transformation_solution.py
│      └── transformation.py
└── tests/
   ├── test_imputation_solution.py
   ├── test_imputation.py
   └── test_transformation.py
</pre>

Data-oriented tests often need to validate more than one number. A transformation can change values, indexes, column names, dtypes, or grouping structure all at once, so this notebook focuses on checks that compare complete pandas objects instead of isolated scalar results.

That difference is important in data work. A pipeline can look correct if you inspect only one cell, while still being wrong because the rows are reordered, missing values were filled incorrectly, or grouped totals were computed on the wrong columns. Testing whole `Series` and `DataFrame` objects makes those problems visible much earlier.

This notebook uses a small jeans-pocket dataset to show two common data-oriented testing patterns:

* testing `Series` outputs with `assert_series_equal`
* testing grouped `DataFrame` outputs with `assert_frame_equal`

**What to expect:** the baseline imputation helper and its test already pass. The transformation exercise is intentionally unfinished, so `tests/test_transformation.py` should fail until you implement `transformation.py`.

> All validation commands in this notebook assume your terminal is currently inside `03-testing-in-data-science/`.


In [ ]:
# Automatically reload modules when they are edited to avoid restarting the kernel.
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path
import sys

for candidate in (
    Path.cwd().resolve(),
    Path.cwd().resolve() / "03-testing-in-data-science",
):
    if (candidate / "src").exists():
        NOTEBOOK_ROOT = candidate
        break
else:
    NOTEBOOK_ROOT = Path.cwd().resolve()

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))


## Setup

We only need a few libraries here: pandas and NumPy for the data examples, and pytest plus pandas testing helpers for the assertions.

The main reason for calling this out is that the testing tools are part of the workflow, not an extra layer added afterwards. In data work, analysis code and validation code usually evolve together.


In [ ]:
import warnings

import numpy as np
import pandas as pd
import pytest
import ipytest
from pandas.testing import assert_frame_equal, assert_series_equal

warnings.filterwarnings("ignore")
ipytest.autoconfig()


Load the dataset and inspect only the columns needed for the examples in this notebook.

Keeping the exploration narrow is deliberate. The goal here is not a full exploratory analysis of the dataset, but a focused look at the columns that support the tested transformations.


In [ ]:
df = pd.read_csv(NOTEBOOK_ROOT / "data/measurementRectanglesMissing.csv")
df[["brand", "menWomen", "price", "pocketArea"]].head()


## Missing Values

A very common data-cleaning step is **imputation**: replacing missing values with a sensible fallback. Here we start with mean imputation for the `price` column.

The reason this is worth testing is that missing-value handling often becomes part of a larger pipeline contract. If the imputation rule changes silently, every downstream calculation that depends on that column can change too.


In [ ]:
# Amount of missing values and percentage of missing values in `price` column.
df["price"].isnull().sum(), round(df["price"].isnull().mean() * 100, 1)


In [ ]:
from src.data_pipeline.imputation import impute

sample_prices = pd.Series([49.0, np.nan, 99.0])
impute(sample_prices)


When you test pandas objects, regular `assert` is often too vague. `assert_series_equal()` gives clearer failures because it compares the full `Series`, not just one value.

That means the test can catch the actual structure of the result: values, length, index alignment, and sometimes dtype-related differences. In data work, those details are often exactly where subtle bugs appear.


In [ ]:
%%ipytest -qq


def test_impute_demo():
    input_series = pd.Series([49.0, np.nan, 99.0])
    expected_result = pd.Series([49.0, 74.0, 99.0])
    assert_series_equal(impute(input_series), expected_result)


## Implementation Target 1: Extend Imputation Helpers

The baseline mean-imputation helper already works. Extend the same file with sibling helpers for min and max imputation.

1. Open `src/data_pipeline/imputation.py`.
2. Keep the existing `impute()` helper as the mean-imputation example.
3. Add `impute_min()` and `impute_max()` by following the same pattern.
4. Run `../.venv/bin/python -m pytest -q tests/test_imputation.py`.
5. Expect that command to pass immediately because it checks the existing baseline helper.
6. Use the solution block and reference checks to compare the additional helpers.


In [ ]:
# @TODO Implementation Target 1: Add min/max imputation helpers.
# Objective: Extend the existing mean-imputation file with min/max variants.
# Edit files:
# - src/data_pipeline/imputation.py
# Validate with:
# - ../.venv/bin/python -m pytest -q tests/test_imputation.py
# Solution:
# - src/data_pipeline/imputation_solution.py


<details>
  <summary>Solution</summary>

Both helpers follow the same pattern: compute one statistic, then use `fillna()`.

```python
import pandas as pd


def impute_min(series: pd.Series) -> pd.Series:
    return series.fillna(series.min())



def impute_max(series: pd.Series) -> pd.Series:
    return series.fillna(series.max())
```
</details>


## Data Transformations

Another common workflow step is creating new features from existing data. Here we turn `pocketArea` into a simple score:

* `0` when a value is at or below the average
* `1` when a value is above the average

This is a small example, but it reflects a broader pattern in data pipelines: raw measurements are often transformed into features that are easier to aggregate, compare, or feed into downstream logic.


In [ ]:
def is_greater_than_average_demo(series: pd.Series) -> pd.Series:
    average = series.mean()
    return pd.Series([0 if value <= average else 1 for value in series])


pocket_area_sample = pd.Series([1000, 2000, 3000, 2500, 4000])
is_greater_than_average_demo(pocket_area_sample)


This time the result is still a `Series`, so `assert_series_equal()` is still the right test helper. Notice that the whole output pattern matters, not just one element.

A transformation can be wrong even if one or two sample values look correct. The test is stronger when it checks the full output against the expected row-by-row pattern.


In [ ]:
%%ipytest -qq


@pytest.mark.parametrize(
    "input_series, expected_result",
    [
        (pd.Series([1, 2, 3, 2.5, 4]), pd.Series([0, 0, 1, 0, 1])),
        (pd.Series([10, 10, 10, 10]), pd.Series([0, 0, 0, 0])),
    ],
)
def test_is_greater_than_average_demo(input_series, expected_result):
    assert_series_equal(is_greater_than_average_demo(input_series), expected_result)


Once you have that row-level score, you often need a grouped summary. Here we group by brand and gender, then sum the score column.

This is a very common pipeline step: create a row-level feature first, then aggregate it into a more compact table for reporting or downstream analysis.


In [ ]:
def get_sum_score_by_brand_and_gender_demo(
    frame: pd.DataFrame,
    brand_col="brand",
    gender_col="menWomen",
    score_by="size_greater_than_average",
) -> pd.DataFrame:
    return frame.groupby(by=[brand_col, gender_col], as_index=False)[score_by].sum()


example_frame = pd.DataFrame(
    {
        "brand": ["Abercrombie", "Abercrombie", "Abercrombie", "Abercrombie"],
        "menWomen": ["men", "men", "women", "women"],
        "size_greater_than_average": [1, 1, 0, 1],
    }
)
get_sum_score_by_brand_and_gender_demo(example_frame)


Now the output is a grouped `DataFrame`, so `assert_frame_equal()` is the better fit. This compares the full table structure, column names, and values.

That matters because grouped data can be wrong in several ways at once: the rows might be missing, the grouping keys might be misaligned, or the aggregate values might be correct but attached to the wrong labels.


In [ ]:
%%ipytest -qq


def test_get_sum_score_by_brand_and_gender_demo():
    input_frame = pd.DataFrame(
        {
            "brand": ["Abercrombie", "Abercrombie", "Abercrombie", "Abercrombie"],
            "menWomen": ["men", "men", "women", "women"],
            "size_greater_than_average": [1, 1, 0, 1],
        }
    )
    expected_result = pd.DataFrame(
        {
            "brand": ["Abercrombie", "Abercrombie"],
            "menWomen": ["men", "women"],
            "size_greater_than_average": [2, 1],
        }
    )
    assert_frame_equal(
        get_sum_score_by_brand_and_gender_demo(input_frame),
        expected_result,
    )


## Implementation Target 2: Implement Transformation Helpers

Move the transformation logic into the repository source file.

1. Open `src/data_pipeline/transformation.py`.
2. Implement `is_greater_than_average()` so it returns one flag per input value.
3. Implement `get_sum_score_by_brand_and_gender()` so it groups by brand and gender and sums the score column.
4. Run `../.venv/bin/python -m pytest -q tests/test_transformation.py`.
5. Expect this command to fail until the stubs are replaced, then pass once both helpers match the tests.


In [ ]:
# @TODO Exercise 2: Implement transformation helpers.
# Objective: Keep row-level flags and grouped score aggregation consistent.
# Edit files:
# - src/data_pipeline/transformation.py
# Validate with:
# - ../.venv/bin/python -m pytest -q tests/test_transformation.py
# Solution:
# - src/data_pipeline/transformation_solution.py


<details>
  <summary>Solution</summary>

```python
import pandas as pd


def is_greater_than_average(series: pd.Series) -> pd.Series:
    average = series.mean()
    return pd.Series([0 if value <= average else 1 for value in series])



def get_sum_score_by_brand_and_gender(
    frame: pd.DataFrame,
    brand_col="brand",
    gender_col="menWomen",
    score_by="size_greater_than_average",
) -> pd.DataFrame:
    return frame.groupby(by=[brand_col, gender_col], as_index=False)[score_by].sum()
```
</details>


## Running Tests

From inside `03-testing-in-data-science/`:

##### Validation commands:

```zsh
../.venv/bin/python -m pytest -q tests/test_imputation.py
../.venv/bin/python -m pytest -q tests/test_transformation.py
```

**What to expect:**

* `tests/test_imputation.py` should already pass, because the baseline mean-imputation helper is already implemented.
* `tests/test_transformation.py` should fail until the incomplete transformation functions are implemented.

##### Reference check:

```zsh
../.venv/bin/python -m pytest -q tests/test_imputation_solution.py
```

##### Run all module tests:

```zsh
../.venv/bin/python -m pytest -q tests
```

Because this notebook intentionally includes unfinished implementation targets, `pytest -q tests` will fail until `transformation.py` is completed.

Use `-s` when you want to see printed output.


### Quiz

1. **When is `assert_series_equal()` more useful than a plain `assert`?**  
   [ ] When you want pytest to skip the test automatically  
   [ ] When you want to compare a full pandas `Series` result  
   [ ] When you are testing only strings  
   [ ] When you want to avoid using pandas  

   <details>
     <summary>Show Answer</summary>

     **Correct Answer:** When you want to compare a full pandas `Series` result  
     **Description:** `assert_series_equal()` checks the full pandas object, not just one value.
   </details>

---

2. **Why use `assert_frame_equal()` for the grouped score output?**  
   [ ] Because grouped results are always strings  
   [ ] Because the output is a pandas `DataFrame`, not a single scalar  
   [ ] Because pytest cannot test grouped data otherwise  
   [ ] Because it automatically fills missing values  

   <details>
     <summary>Show Answer</summary>

     **Correct Answer:** Because the output is a pandas `DataFrame`, not a single scalar  
     **Description:** `assert_frame_equal()` compares the full table structure and values.
   </details>
